In [43]:
with open("sentences.txt", "r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f if line.strip()]


In [44]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_vectors = model.encode(sentences).astype('float32')


In [45]:
id_to_sentence = {i: sentence for i, sentence in enumerate(sentences)}


In [46]:
import faiss

dim = sentence_vectors.shape[1]
index = faiss.IndexFlatL2(dim)  # L2 distance (Euclidean)
index.add(sentence_vectors)     # Add all sentence vectors


In [59]:
query = "call police"
query_vector = model.encode([query]).astype('float32')




In [60]:
%%time
# Search for top 5 similar sentences
D, I = index.search(query_vector, k=5)

print(I)

# Print results
print("Query:", query)
print("\nTop Matches:")
for i, dist in zip(I[0], D[0]):
    print(f"- {id_to_sentence[i]}  (distance: {dist:.4f})")

[[  536   171 14000 10294 13805]]
Query: call police

Top Matches:
- Security lights have also been installed and police have swept the grounds for booby traps.  (distance: 0.9099)
- The Rev. John Johnston, 64, was charged with aggravated harassment in the phone call case and with criminal possession of a weapon, according to a police statement.  (distance: 0.9232)
- refrain from harming.  (distance: 0.9351)
- have unlawful sex with a prostitute  (distance: 0.9492)
- Special police detained Khodorkovsky early on Saturday in the Siberian city of Novosibirsk, where his plane had made a refuelling stop.  (distance: 0.9571)
CPU times: total: 0 ns
Wall time: 1.15 ms


In [61]:
quantizer = faiss.IndexFlatL2(dim)

In [62]:
nlist = 50                     # number of clusters (coarse quantizer)
m = 8                          # number of sub-vectors for PQ
nbits = 8                      # bits per sub-vector
index = faiss.IndexIVFPQ(quantizer, dim, nlist, m, nbits)


In [63]:
index.is_trained

False

In [64]:
index.train(sentence_vectors)

In [65]:
index.add(sentence_vectors)

In [66]:
query = "call police"
query_vec = model.encode([query]).astype('float32')

# Set number of clusters to probe during search (tradeoff: accuracy vs speed)
index.nprobe = 10




In [67]:
%%time
# Search top 5
D, I = index.search(query_vec, k=5)
print(I)
print("privous result:[[1966 2017 5092 9010 4240]]")
#[[9582 8537 2862 9360 3030]]

[[  536   171 14000 10294 13805]]
privous result:[[1966 2017 5092 9010 4240]]
CPU times: total: 0 ns
Wall time: 0 ns


In [56]:
print(f"Query: {query}\n")
print("Top Matches:")
for idx, dist in zip(I[0], D[0]):
    print(f"- {id_to_sentence[idx]}  (distance: {dist:.4f})")

Query: a woman is jumping into water!

Top Matches:
- A woman is diving into a pool  (distance: 0.4505)
- A woman in a blue bathing suit is jumping off a dock into a lake.  (distance: 0.5582)
- A lady is surfing and riding a wave  (distance: 0.5642)
- A woman in a yellow dyed shirt is surfing on a pink surfboard  (distance: 0.6013)
- A woman in a yellow shirt is surfing on a pink surfboard  (distance: 0.6013)
